# 목차
* [Chapter 1 주요 개념에 대한 객관식 질문](#chapter1)
* [Chapter 2-1 LLM구조 구현하기](#chapter2_1)
* [Chapter 2-2 층 정규화로 활성화 정규화하기](#chapter2_2)
* [Chapter 2-3 GELU 활성화 함수를 사용하는 피드 포워드 네트워크 구현하기](#chapter2_3)
* [Chapter 2-4 숏컷 연결 추가하기](#chapter2_4)
* [Chapter 2-5 어텐션과 선형층을 트렌스포머 블록에 연결하기](#chapter2_5)
* [Chapter 2-6 GPT 모델 만들기](#chapter2_6)
* [Chapter 2-7 텍스트 생성하기](#chapter2_7)

## Chapter 1 주요 개념에 대한 객관식 질문 <a class="anchor" id="chapter1"></a>
1. GPT_CONFIG_124M 딕셔너리에서 context_length의 목적은 무엇인가요?
   - d. 모델이 위치 임베딩으로 다룰 수 있는 입력 토큰의 최대 개수를 나타냅니다.

2. GTP 모델에 있는 층 정규화의 주요 목적은 신경망 층의 활성화를 조정하여 평규이 0이고 ___가(이) 1이 되도록 만드는 것입니다.
   - a. 분산

3. LLM에서 배치 정규화 대신 층 정규화를 사용하는 이점은 무엇인가요?
   - b. 층 정규화는 배치에 있는 입력을 독립적으로 정규화합니다.

4. GPT-2와 같은 LLM에서 ReLU의 대안으로 널리 사용되는 활성화 함수는 무엇인가요?
    - b. GELU

5. 심층 신경망에서 숏컷 연경의 주요 목적은 무엇인가요?
   - b. 훈련할 때 역전파 과정에서 그레이디언트 흐름을 보존하기 위해

6. GPT 모델에서 트랜스포머 블록의 주요 구성 요소는 무엇인가요?
   - c. 멀티 헤드 어텐션, 층 정규화, 드롭아웃, 피드 포워드 층, GELU 활성화 함수

7. 훈련하지 않은 GPT 모델이 횡성수설하는 이유는 무엇인가요?
   - c. 모델이 단어 사이의 관계와 언어에 있는 패턴을 학습하지 않았습니다.

## Chapter 2-1 LLM구조 구현하기 <a class="anchor" id="chapter2_1"></a>
1. GPT 모델의 주요 목적은 무엇인가요? 어떻게 이를 달성하나요?
   - 목표 : 주어진 텍스트 시퀸스에서 다음 토큰을 예측한다.
   - 달성 방법: 토큰화된 텍스트 -> 입베딩 층 -> 트랜스포머 블록 -> 출력층 -> 다음 토큰 예측

2. 아래의 그림에서 번호에 들어갈 이름을 넣으세요.
    - 1: 출력 층
    - 2: 임베딩 층
    - 3: 토큰화된 텍스트

        ![W-04-01-02](image/W-04-01-02.png)

3. GPT 모델의 주요 구성 요소는 무엇인가요? 모델의 기능에 어떤 기여를 하나요?
    - 토큰화된 텍스트: 원시 텍스트를 토큰 단위로 분할하여 모델이 처리할 수 있는 형태로 변환
    - 임베딩 층: 토큰을 고차원 벡터로 변환하여 모델이 의미를 이해할 수 있도록 함
    - 트랜스포머 블록: 셀프 어텐션 메커니즘과 피드포워드 신경망을 사용하여 입력 시퀀스 내의 관계를 학습
    - 출력 층: 트랜스포머 블록의 출력을 바탕으로 다음 토큰의 확률 분포를 생성하여 예측 수행

4. GPT와 같은 LLM에서 '파라미터'의 개념을 설명하세요.
   - 파라미터는 모델이 학습하는 가중치와 편향을 의미하며, 모델의 예측 성능에 직접적인 영향을 미친다. 
   - 이들은 입력 데이터를 처리하고 패턴을 학습하는 데 사용된다.

5. GPT-2와 GPT-3 사이의 주요한 차이점은 무엇인가요? LLM을 배울 때 GPT-2를 선택하는 게 더 나은 이유가 무엇인가요?
   - GPT-3는 GPT-2보다 훨씬 더 많은 파라미터를 가지고 있어 더 복잡한 언어 패턴을 학습할 수 있다. 
   - 그러나 GPT-2는 상대적으로 단순하고 이해하기 쉬우며, 학습 자원이 적게 들기 때문에 LLM의 기본 개념을 배우기에 더 적합하다.

6. GPT_CONFIG_124M 딕셔너리의 목적과 주요 키-값 쌍을 설명하세요.
    - 목적: GPT 모델의 구조와 하이퍼파라미터를 정의하여 모델 초기화에 사용
    - 주요 키-값 쌍:
      - vocab_size: 어휘 사전의 크기 (예: 50257)
      - context_length: 최대 시퀀스 길이 (예: 1024)
      - emb_dim: 임베딩 차원 (예: 768)
      - n_layer: 트랜스포머 블록의 수 (예: 12)
      - n_head: 어텐션 헤드의 수 (예: 12)
      - dropout: 0.1 드롭아웃 비율 - 과대적합을 막기 위해 10%를 랜던하게 제외한다.
      - qkv_bias: False 쿼리, 키, 값 선형 변환에 바이어스 항을 사용할지 여부 - 훈련 시 False, 추론 시 True

7. DummyGPTModel의 역활은 무엇인가요? GPT 모델의 전반적인 구현에 어떻게 기여하나요?
   - DummyGPTModel은 GPT 모델의 기본 구조를 단순화하여 구현한 것으로, 주요 구성 요소를 포함하고 있다.
   - 이를 통해 GPT 모델의 작동 원리를 이해하고, 실제 모델 구현에 필요한 기초를 제공한다.


## Chapter 2-2 층 정규화로 활성화 정규화하기 <a class="anchor" id="chapter2_2"></a>
1. 신경망에서 층 정규화의 주요 목적은 무엇인가요? 훈련 과정을 계선하는 데 어떻게 기여하나요?
   - 목적: 신경망 훈련을 안정화하고 가속화하는 것이 목적이다.
   - 기여: 많은층을 가진 신경망 훈련 시 그레이디언트 소실과 폭주 문제를 완화한다.

2. GPT-2와 현대 트랜스포머 구조에서 층 정규화가 적용되는 위치를 설명하세요.
    - 멀티 헤드 어텐션 모듈 전후에 적용된다. 최종 출력층 전에도 적용된다.

3. 다음 코드에서 일부 내용이 삭제되었습니다. 삭제된 코드는 무엇이며 어디에 들어가야 하나요?
    - 1: d. one
    - 2: f. zero
    - 3: c. norm_x

        ![W-04-02-03](image/W-04-02-03.png)

4. 편향된 분산 계산과 편향되지 않은 분산 계산의 차이를 설명하세요. LLM에서 편향된 계산 방식이 선호되는 이유는 무엇인가요?
    - 차이점: 편향된 분산 계산은 표본의 크기로 나누고, 편향되지 않은 분산 계산은 (표본의 크기 - 1)로 나눈다.    
    - 이유: LLM에서 임베딩 차원 n은 매우 크므로 편향된 계산은 계산이 더 간단하고, 대규모 데이터셋에서 충분히 정확한 추정치를 제공하기 때문에 LLM에서 선호된다.
           
5. 층 정규화와 배치 정규화의 주요 차이점은 무엇인가요? LLM에서 층정규화를 선호하는 이유가 무엇인가요?
    - 차이점: 층 정규화는 각 샘플의 특성 차원에 대해 정규화하고, 배치 정규화는 배치 내의 샘플들에 대해 정규화한다.
    - 이유: LLM은 가변 길이의 시퀀스를 처리하고, 배치 크기가 작거나 1일 수 있기 때문에 층 정규화가 더 적합하다.

6. 왼쪽의 용어와 오른쪽의 설명을 맞게 연결하세요

    ![W-04-02-06](image/W-04-02-06.png)

## Chapter 2-3 GELU 활성화 함수를 사용하는 피드 포워드 네트워크 구현하기 <a class="anchor" id="chapter2_3"></a>
1. GELU 활성화 함수가 무엇인가요? ReLU 활성화 함수와 어떻게 다른가요?
   - GELU 활성화 함수는 입력값이 정규 분포를 따른다고 가정하에 설계된 활성화 함수로, ReLU보다 부드러운 곡선을 가지고있다.
   - ReLU 활성화 함수는 음수는 0으로, 양소는 그대로 출력하는 반면, GELU는 x=-075에서 출력이 0.2 정도로 신경망이 음수에서도 학습 할 수 있도록 돕는다.

2. LLM에 있는 FeedForwar 모듈의 목적과 구조에 대해 설명하세요.
   - 목적
      -  각 토큰의 표현을 변환하고 강화하는 역할을 한다.
   - 구조
      - 일반적으로 선형변화 2개와 활성화 함수 1개로 이루어지며, 각 토큰의 표현을 변환하고 강화하는 역할을 한다.
      - 첫 번째 선형 변환은 입력 차원을 확장하고, 활성화 함수를 통과한 후, 두 번째 선형 변환 후 원래 차원으로 축소한다.

3. 다음 코드에서 일부 내용이 삭제되었습니다. 삭제된 코드는 무엇이며 어디에 들어가야하나요?
    - 1: e. Sequential
    - 2: d. GELU

        ![W-04-03-03](image/W-04-03-03.png)

4. FeedForward 모듈이 모델의 학습 능력과 일반화 능력에 어떻게 기여하나요?
   - 학습 능력: 더 복잡한 패턴을 학습할 수 있도록 모델의 표현력을 향상시킨다.
   - 일반화 능력: 각 토큰의 표현을 더욱 풍부하게 만들어준다.

5. FeedForward 모둘이 동일한 입력과 출력 차원을 갖는 이유는 무엇인가요?
   - 입력 차원과 출력 차원이 동일하기 때문에 여러 층을 쌓아도 시퀀스 길이나 토큰 표현의 차원이 변하지 않는다.
   - 모델의 규모를 쉽게 늘릴 수 있다.


## Chapter 2-4 숏컷 연결 추가하기 <a class="anchor" id="chapter2_4"></a>
1. 그레이디언트 소실 문제가 무엇이고, 심층 신경망 훈련에 어떤 영향을 미치나요?
   - 그레이디언트가 심층 신경망의 층을 통해 역전파되면서 점점 작아질 때 일어나며 앞에 있는 층을 효과적으로 훈련하기 어렵다.
   - 이는 모델의 학습 과정을 방해하고 최적의 성능을 달성하지 못하게 한다.

2. 숏컷 연결의 개념과 그레이디언트 소실 문제를 어떻게 해결하는지 설명하세요.
   - 그레이디언트가 특정 층을 우회해서 신경망에 흐를 수 있는 대안 경로를 만든다.
   - 이를 통해 그레이디언트가 소실되는 것을 방지하고, 초기층이 더 쉽게 학습할 수 있도록 돕는다.

3. 다음 코드에서 숏컷 연결이 어떻게 구현되나요?
    - 클래스 생성 시, use_shortcut 매개변수가 True이고 입력 shape와 출력 shape가 동일할때 출력에 입력을 더한다.

        ![W-04-04-03](image/W-04-04-03.png)

4. print_gradients 함수의 목적은 무엇이고, 어떻게 숏컷 연결 효과를 보여주나요?
   - 목적: 모델의 각 층에서 그레이디언트의 크기를 출력하여 숏컷 연결이 그레이디언트 소실 문제를 완화하는지 확인한다.
   - 효과: 숏컷 연결이 있을 때 그레이디언트 크기가 더 크게 유지되는 것을 보여준다.

5. 다음 그림은 다섯 개의 층으로 구성된 심층 신경망(왼쪽)과 숏컷 연결을 사용하는 신경망(오른쪽)을 보여줍니다. 오른쪽 신경망의 그레이디언트 값이 더 큰이유는 무엇인가요?
   - 층을 통과한 후 출력에 입력을 더하기 때문에 그레이디언트가 소실되지 않고 더 크게 유지된다.

      ![W-04-04-05](image/W-04-04-05.png)

## Chapter 2-5 어텐션과 선형층을 트렌스포머 블록에 연결하기<a class="anchor" id="chapter2_5"></a>
1. 트랜스포머 블록의 핵심 구성 요소는 무엇인가요? 입력 시퀴스를 처리하는 데 어떻게 기여하나요?
   - 주요구성 요소: 층정규화, 멀티 헤드 어텐션, 드롭 아웃, 피드포워드 네트워크, 숏컷 연결
   - 멀티 헤드 어텐션: 입력 시퀸스에 있는 각 토큰 간의 관계를 분석한다.
   - 피드 포워드 네트워크: 각 위치의 데이터를 개별적으로 변형한다.
   - 층 정규화: 출력의 크기를 일정하게 유지시킨다.
   - 드롭 아웃: 과대적합을 막는다.
   - 숏컷 연결: 그레이디언트 소실 문제를 완화한다.

2. 사전 층 정규화 개념과 트랜스포머 불록에서 중요성을 설명하세요.
   - 사전 층 정규화는 각 서브층(어텐션, 피드포워드 네트워크) 전에 입력을 정규화하는 기법이다.
   - 중요성: 훈련 안정성을 높이고, 그레이디언트 소실 문제를 완화하여 심층 모델의 학습을 용이하게 한다.

3. 트랜스포머 블록에서 숏컷 연결의 역활과 그레이디언트 흐름에 대한 영향을 설명하세요.
   - 역활: 입력을 출력에 더하여 그레이디언트가 특정 층을 우회할 수 있도록 한다.
    - 영향: 그레이디언트 소실 문제를 완화하고, 초기 층이 더 효과적으로 학습할 수 있도록 돕는다.

4. 트랜스포머 블록에서 입력 차원을 출력에 어떻게 유지하나요?
   - 피드포워드 네트워크의 첫 번째 선형 변환에서 입력 차원을 확장하고, 두 번째 선형 변환에서 원래 차원으로 축소한다.
  
5. 트랜스포머 블록이 모든 입력 시퀸스의 문맥 정보를 출력에 통합하는 방법을 설명하세요.
    - 멀티 헤드 셀프 어텐션 메커니즘을 사용하여 각 토큰이 시퀸스 내의 다른 모든 토큰과 상호작용할 수 있도록 한다.
    - 이를 통해 각 토큰의 표현이 전체 시퀸스의 문맥 정보를 반영하도록 한다.


## Chapter 2-6 GPT 모델 만들기<a class="anchor" id="chapter2_6"></a>
1. GPTModel 클래스의 목적은 무엇인가요? TransformerBlock 클래스와 어떤 연관이 있나요?
   - 목적: GPT 모델의 전체 구조를 정의하고, 입력 텍스트를 처리하여 다음 토큰을 예측한다.
   - 연관성: GPTModel 클래스는 GPT 모델 생성 시 TransformerBlock 클래스를 12번 인스턴스화 하여 트랜스포머 블록을 쌓아 올린다.

2. GPTModel 구조에서 LayerNorm 층의 역활을 설명하세요
   - LayerNorm은 층 정규화를 수행한다
   - 훈련을 안정화 시키고 수렴 속도를 높인다

3. 가중치 묶기가 무엇이며, GPT 모델의 파라미터 개수에 어떻게 영향을 미치나요?
    - 가중치 묶기는 입력 임베딩 층과 출력 층에서 동일한 가중치 행렬을 공유하는 방법이다.
    - 토큰 임베딩 층은 50,257차원의 원-핫 인코딩된 입력 토큰을 768차원의 임베딩 표현에 투영합니다.
    - 출력 층은 768차원의 임베딩을 단어로 변환하기 위해 50,257차원의 표현으로 다시 투영합니다
    - 따라서 가중치 행렬의 크기를 보면 알 수 있듯이 임베딩 층과 출력 층의 파라미터 개수가 같습니다
    - 이를 통해 모델의 메모리 사용량을 줄이고, 훈련 속도를 향상시킬 수 있다.

4. GTPModel 출력을 텍스트로 변환하는 과정을 설명하세요.
    - GPTModel의 출력은 각 토큰에 대한 다음 토큰의 확률 분포를 나타내는 로짓 벡터이다.
    - 이 로짓 벡터에 소프트맥스 함수를 적용하여 확률 분포로 변환한다.
    - 그런 다음, 확률이 가장 높은 토큰을 선택하여 텍스트로 변환한다.

5. 다음 코드에서 일부 내용이 삭제되었습니다. 삭제된 코드는 무엇이며 어디에 들어가야하나요?
    - 1: b. Dropout
    - 2: e. Sequential
    - 3: d. Linear

        ![W-04-06-05](image/W-04-06-05.png)

6. 트랜스포머 블록의 개수가 GPT 모델의 복잡도와 성능에 어떤 영향을 미치나요?
    - 트랜스포머 블록의 개수가 많을수록 모델의 복잡도가 증가하여 더 복잡한 언어 패턴을 학습할 수 있다.
    - 그러나 블록 수가 너무 많으면 과대적합 위험이 증가하고, 훈련 시간이 길어질 수 있다.
    - 적절한 블록 수를 선택하는 것이 모델 성능 최적화에 중요하다.

7. 왼쪽의 용어와 오른쪽의 설명을 맞게 연결하세요

    ![W-04-06-07](image/W-04-06-07.png)

## Chapter 2-7 텍스트 생성하기<a class="anchor" id="chapter2_7"></a>
1. 출력 텐서부터 시작해서 GPT 모델이 텍스트를 생성하는 과정을 설명하세요.
   - GPT 모델은 입력의 샘플의 개수와 같은 개수의 출력과 각 출력이 어휘 사전의 크기인 50,257 차원을 가진 로짓 텐서를 생성한다.
   - 다음 토큰에 해당하는 마지막 벡터를 추출한다.
   - 소프트맥스 함수를 사용하여 확률 분포로 변환한다.
   - 가장 큰 값의 인덱스 위치(토큰 ID)를 찾는다.
   - 해당 토큰 ID를 텍스트로 변환한다.

2. 텍스트 생성 과정에서 소프트맥스 함수의 역활을 설명하세요.
   - 소프트맥스 함수는 로짓 벡터를 확률 분포로 변환하여 각 토큰이 다음 토큰으로 선택될 확률을 나타낸다.
   - 이를 통해 모델이 가장 가능성 높은 토큰을 선택할 수 있도록 한다.

3. generate_text_simple 함수의 목적은 무엇이며 어떻게 작동하나요?
   - 목적: GPT 모델을 사용하여 주어진 텍스트로 부터 시작하여 새로운 텍스트를 생성한다.
   - 작동 방식: 입력 텐서를 모델에 전달, 모델이 출력한 로짓에서 다음 토큰을 선택, 선택된 토큰을 입력에 추가하여 반복적으로 텍스트를 생성한다.

4. generate_text_simple 함수에서 softmax 단계가 중복인 이유는 무엇인가요?
   - GPT 모델의 출력 로짓을 확률 분포로 변환하지 않아도, 가장 큰 값의 인덱스를 찾는 데는 영향을 미치지 않기 때문이다.

5. 텍스트 생성에서 그리디 디코딩의 의미는 무엇인가요? 
   - 의미: 매 단계에서 가장 확률이 높은 토큰을 선택하여 텍스트를 생성하는 방법이다.
   - 장점: 구현이 간단하고 빠르다.
   - 단점: 다양성이 부족하고, 반복적인 패턴이 나타날 수 있다.

6. 훈련되지 않은 GPT 모델이 횡성수설하는 이유는 무엇인가요?
    - 훈련되지 않은 GPT 모델은 언어 패턴과 문법을 학습하지 못했기 때문에 무작위로 토큰을 선택하여 일관성 없는 출력을 생성한다.

7. GPTModel 클래스를 구현하는 단계를 올바른 순서로 나열하세요.
   - 1: d. nn.Module을 상속한 GPTModel 클래스를 만든다.
   - 2: a. 토큰 임베딩, 위치 임베딩, 드롭아웃, 선형 출력층을 초기화한다.
   - 3: c. 여러개의 TransformerBlock 인스턴스를 담는 Sequential 객체를 만든다.
   - 4: b. 정방향 계산, 임베딩 결합, 트랜스포머 블록, 층 정규화, 출력층을 초기화한다.
